# Advanced Problems with Solutions: Python Mutable Sequences

This notebook contains advanced practice problems on Python mutable sequences, especially lists.

Topics covered:

- in-place mutation vs rebinding
- aliasing
- replacing elements
- slice assignment
- `append` vs concatenation
- `extend` and iterable ordering
- `pop`, `insert`, and removal patterns
- in-place reversal vs reversed copies
- shallow copying and aliasing traps

Recommended workflow:

1. Read the problem.
2. Try to solve it before looking at the solution.
3. Run the tests.
4. Modify the examples to check edge cases.

## Problem 1 — Mutation vs Rebinding

Two names may refer to the same mutable object.

Consider this setup:

```python
items = [1, 2, 3, 4]
alias = items
```

### Tasks

1. Predict what happens after `items.clear()`.
2. Predict what happens after `items = []`.
3. Write a function `empty_in_place(seq)` that empties a mutable sequence without rebinding the caller's object.
4. Write tests proving that aliases observe the mutation.

In [1]:
# Demonstration: clear() mutates the existing list

items = [1, 2, 3, 4]
alias = items

before_id = id(items)
items.clear()
after_id = id(items)

print("items:", items)
print("alias:", alias)
print("same object:", before_id == after_id)

assert items == []
assert alias == []
assert before_id == after_id
assert items is alias

items: []
alias: []
same object: True


In [2]:
# Demonstration: rebinding changes only the local name

items = [1, 2, 3, 4]
alias = items

before_id = id(items)
items = []
after_id = id(items)

print("items:", items)
print("alias:", alias)
print("same object:", before_id == after_id)

assert items == []
assert alias == [1, 2, 3, 4]
assert before_id != after_id
assert items is not alias

items: []
alias: [1, 2, 3, 4]
same object: False


In [3]:
def empty_in_place(seq):
    """Empty a mutable sequence without rebinding it."""
    seq.clear()


data = [10, 20, 30]
alias = data
old_id = id(data)

empty_in_place(data)

assert data == []
assert alias == []
assert id(data) == old_id
assert data is alias

print("All tests passed.")

All tests passed.


### Solution notes

`clear()` mutates the object. Rebinding with `items = []` makes the name `items` point to a new list, while other aliases still point to the old list.

## Problem 2 — Append vs Concatenation

`append` mutates a list in place. Concatenation with `+` creates a new list.

### Task

Write two functions:

1. `add_item_in_place(seq, item)` — mutates the original list.
2. `add_item_copy(seq, item)` — returns a new list and leaves the original unchanged.

Then prove the difference using `id` and aliases.

In [4]:
def add_item_in_place(seq, item):
    """Append item to seq and return the same list object."""
    seq.append(item)
    return seq


def add_item_copy(seq, item):
    """Return a new list with item added, leaving seq unchanged."""
    return seq + [item]


original = [1, 2, 3]
alias = original
old_id = id(original)

result = add_item_in_place(original, 4)

assert result is original
assert original is alias
assert id(original) == old_id
assert alias == [1, 2, 3, 4]

print("in-place result:", result)

in-place result: [1, 2, 3, 4]


In [5]:
original = [1, 2, 3]
alias = original
old_id = id(original)

result = add_item_copy(original, 4)

assert result is not original
assert original is alias
assert id(original) == old_id
assert original == [1, 2, 3]
assert result == [1, 2, 3, 4]

print("original:", original)
print("result:  ", result)

original: [1, 2, 3]
result:   [1, 2, 3, 4]


### Best-practice takeaway

Use `append` when the caller expects mutation. Use concatenation or unpacking when the caller expects a new list.

## Problem 3 — Slice Assignment With Length Changes

Mutable sequences can replace a slice with a sequence of a different length.

Example:

```python
l = [1, 2, 3, 4, 5]
l[0:2] = ['a', 'b', 'c', 'd']
```

### Task

Write `replace_middle(seq, replacement)` that replaces everything except the first and last elements of `seq` with `replacement`.

Requirements:

- Mutate `seq` in place.
- Work for lists of length 2 or more.
- Raise `ValueError` for lists shorter than 2.
- Return the same object that was passed in.

In [6]:
def replace_middle(seq, replacement):
    """Replace all elements except first and last with replacement."""
    if len(seq) < 2:
        raise ValueError("seq must contain at least two elements")

    seq[1:-1] = replacement
    return seq


data = [1, 2, 3, 4, 5]
alias = data
old_id = id(data)

result = replace_middle(data, ['a', 'b', 'c'])

assert result is data
assert data is alias
assert id(data) == old_id
assert data == [1, 'a', 'b', 'c', 5]

print(data)

[1, 'a', 'b', 'c', 5]


### Solution notes

Slice assignment can grow, shrink, or preserve the length of a list. The object identity stays the same because the list is mutated in place.

## Problem 4 — Extend With Any Iterable

`extend` accepts any iterable, not just lists.

### Task

Write `extend_report(seq, iterable)` that mutates `seq` by extending it with `iterable`, and returns a report containing:

- the original object id
- the final object id
- whether identity was preserved
- how many items were added
- the final list

Then test it with a tuple, a string, a set, and a generator.

In [7]:
def extend_report(seq, iterable):
    """Extend seq with iterable and report what happened."""
    before_id = id(seq)
    before_len = len(seq)

    seq.extend(iterable)

    after_id = id(seq)
    after_len = len(seq)

    return {
        "before_id": before_id,
        "after_id": after_id,
        "identity_preserved": before_id == after_id,
        "items_added": after_len - before_len,
        "final": seq
    }


examples = [
    ([], (1, 2, 3)),
    ([], "abc"),
    ([], {"x", "y", "z"}),
    ([], (n * n for n in range(4)))
]

for seq, iterable in examples:
    report = extend_report(seq, iterable)
    print(report)
    assert report["identity_preserved"] is True

{'before_id': 2969099195072, 'after_id': 2969099195072, 'identity_preserved': True, 'items_added': 3, 'final': [1, 2, 3]}
{'before_id': 2969099142272, 'after_id': 2969099142272, 'identity_preserved': True, 'items_added': 3, 'final': ['a', 'b', 'c']}
{'before_id': 2969099194624, 'after_id': 2969099194624, 'identity_preserved': True, 'items_added': 3, 'final': ['x', 'z', 'y']}
{'before_id': 2969099357568, 'after_id': 2969099357568, 'identity_preserved': True, 'items_added': 4, 'final': [0, 1, 4, 9]}


### Important note

Extending with a set does not guarantee a meaningful positional order. Extending with ordered iterables such as tuples, lists, strings, ranges, and generators preserves their iteration order.

## Problem 5 — Safe Pop From Either End

`pop()` removes and returns an element. Without an index, it removes the last item.

### Task

Write `safe_pop(seq, index=-1, default=None)`.

Requirements:

- Mutate `seq` only when the index is valid.
- Return `default` if the list is empty or the index is invalid.
- Support negative indexes using normal Python semantics.
- Do not catch unrelated exceptions silently.

In [8]:
def safe_pop(seq, index=-1, default=None):
    """Pop seq[index] if valid, otherwise return default."""
    try:
        return seq.pop(index)
    except IndexError:
        return default


data = [10, 20, 30]

assert safe_pop(data) == 30
assert data == [10, 20]

assert safe_pop(data, 0) == 10
assert data == [20]

assert safe_pop(data, 99, default="missing") == "missing"
assert data == [20]

assert safe_pop([], default="empty") == "empty"

print("All tests passed.")

All tests passed.


### Solution notes

Catching `IndexError` is appropriate here because it is exactly the failure mode for an invalid pop index.

## Problem 6 — Insert While Preserving Sorted Order

`insert(index, value)` mutates a list by inserting an item before the given index.

### Task

Write `insert_sorted(seq, value)` that inserts `value` into an already sorted list while preserving sorted order.

Requirements:

- Mutate `seq` in place.
- Return the inserted index.
- Do not use `sort` after inserting.
- Work with duplicate values by inserting after existing equal values.

In [9]:
def insert_sorted(seq, value):
    """Insert value into sorted seq after existing equal values."""
    index = 0

    while index < len(seq) and seq[index] <= value:
        index += 1

    seq.insert(index, value)
    return index


data = [1, 2, 2, 4, 5]
old_id = id(data)

idx = insert_sorted(data, 2)

assert idx == 3
assert data == [1, 2, 2, 2, 4, 5]
assert id(data) == old_id

idx = insert_sorted(data, 0)
assert idx == 0
assert data == [0, 1, 2, 2, 2, 4, 5]

idx = insert_sorted(data, 99)
assert idx == len(data) - 1
assert data == [0, 1, 2, 2, 2, 4, 5, 99]

print("All tests passed.")

All tests passed.


### Advanced note

For large sorted lists, Python's `bisect` module is more efficient and idiomatic. This problem implements the logic manually to clarify how insertion indexes work.

## Problem 7 — Reverse In Place or Return a Reversed Copy?

`reverse()` mutates a list in place. Slicing with `[::-1]` returns a new reversed list.

### Task

Write a function `reverse_control(seq, in_place=False)`.

Requirements:

- If `in_place=True`, mutate the list and return the same object.
- If `in_place=False`, return a reversed copy and leave the original unchanged.
- Use tests to prove the identity behavior.

In [10]:
def reverse_control(seq, in_place=False):
    """Reverse seq in place or return a reversed copy."""
    if in_place:
        seq.reverse()
        return seq

    return seq[::-1]


data = [1, 2, 3, 4]
alias = data
old_id = id(data)

result = reverse_control(data, in_place=True)

assert result is data
assert data is alias
assert id(data) == old_id
assert data == [4, 3, 2, 1]

print("in-place:", data)

in-place: [4, 3, 2, 1]


In [11]:
data = [1, 2, 3, 4]
alias = data
old_id = id(data)

result = reverse_control(data, in_place=False)

assert result is not data
assert data is alias
assert id(data) == old_id
assert data == [1, 2, 3, 4]
assert result == [4, 3, 2, 1]

print("original:", data)
print("copy:    ", result)

original: [1, 2, 3, 4]
copy:     [4, 3, 2, 1]


### Best-practice takeaway

Function names and parameters should make mutation behavior obvious. Silent mutation surprises callers.

## Problem 8 — Shallow Copy Trap

`copy()` and `[:]` create shallow copies. The outer list is copied, but nested mutable objects are shared.

### Task

Given:

```python
matrix = [[1, 2], [3, 4]]
copy1 = matrix.copy()
copy2 = matrix[:]
```

1. Prove that `matrix`, `copy1`, and `copy2` are different outer lists.
2. Prove that their inner lists are shared.
3. Write `copy_matrix(matrix)` that copies both the outer list and each row.

In [12]:
matrix = [[1, 2], [3, 4]]
copy1 = matrix.copy()
copy2 = matrix[:]

print("outer identity:")
print(matrix is copy1)
print(matrix is copy2)

print("inner identity:")
print(matrix[0] is copy1[0])
print(matrix[0] is copy2[0])

assert matrix is not copy1
assert matrix is not copy2
assert matrix[0] is copy1[0]
assert matrix[0] is copy2[0]

copy1[0][0] = 99

assert matrix == [[99, 2], [3, 4]]
assert copy2 == [[99, 2], [3, 4]]

print(matrix)
print(copy1)
print(copy2)

outer identity:
False
False
inner identity:
True
True
[[99, 2], [3, 4]]
[[99, 2], [3, 4]]
[[99, 2], [3, 4]]


In [13]:
def copy_matrix(matrix):
    """Copy a two-dimensional list one row at a time."""
    return [row.copy() for row in matrix]


matrix = [[1, 2], [3, 4]]
copied = copy_matrix(matrix)

assert copied == matrix
assert copied is not matrix
assert copied[0] is not matrix[0]

copied[0][0] = 99

assert copied == [[99, 2], [3, 4]]
assert matrix == [[1, 2], [3, 4]]

print("original:", matrix)
print("copied:  ", copied)

original: [[1, 2], [3, 4]]
copied:   [[99, 2], [3, 4]]


### Solution notes

A shallow copy is enough for flat lists. For nested mutable data, you often need a deeper copying strategy.

## Problem 9 — Mutating While Iterating

Removing items from a list while iterating over it can skip elements or produce confusing behavior.

### Task

Write `remove_in_place(seq, predicate)` that removes all elements matching `predicate` while preserving the identity of `seq`.

Requirements:

- Mutate the original list.
- Do not skip elements.
- Preserve the original order of kept elements.
- Return the same list object.

In [14]:
def remove_in_place(seq, predicate):
    """Remove all items satisfying predicate while preserving list identity."""
    seq[:] = [item for item in seq if not predicate(item)]
    return seq


data = [1, 2, 3, 4, 5, 6]
alias = data
old_id = id(data)

result = remove_in_place(data, lambda x: x % 2 == 0)

assert result is data
assert data is alias
assert id(data) == old_id
assert data == [1, 3, 5]
assert alias == [1, 3, 5]

print(data)

[1, 3, 5]


### Why slice assignment is useful here

`seq = [...]` would rebind the local name and leave aliases unchanged. `seq[:] = [...]` replaces the contents of the original list object.

## Problem 10 — Transactional List Update

Sometimes you want to mutate a list only if every operation succeeds.

### Task

Write `transactional_update(seq, operations)`.

Each operation is a tuple:

```python
("append", value)
("insert", index, value)
("pop", index)
("replace", index, value)
```

Requirements:

- Apply all operations to a temporary copy first.
- If all operations succeed, mutate the original list in place.
- If any operation fails, leave the original list unchanged.
- Return `True` if committed, `False` otherwise.

In [15]:
def transactional_update(seq, operations):
    """Apply list operations atomically from the caller's perspective."""
    working = seq.copy()

    try:
        for op in operations:
            command = op[0]

            if command == "append":
                _, value = op
                working.append(value)

            elif command == "insert":
                _, index, value = op
                working.insert(index, value)

            elif command == "pop":
                _, index = op
                working.pop(index)

            elif command == "replace":
                _, index, value = op
                working[index] = value

            else:
                raise ValueError(f"unknown operation: {command}")

    except Exception:
        return False

    seq[:] = working
    return True


data = [1, 2, 3]
alias = data
old_id = id(data)

ok = transactional_update(data, [
    ("append", 4),
    ("insert", 0, 0),
    ("replace", 2, "x")
])

assert ok is True
assert data == [0, 1, "x", 3, 4]
assert alias == [0, 1, "x", 3, 4]
assert id(data) == old_id

print("committed:", data)

committed: [0, 1, 'x', 3, 4]


In [16]:
data = [1, 2, 3]
alias = data
old_id = id(data)

ok = transactional_update(data, [
    ("append", 4),
    ("replace", 99, "boom")
])

assert ok is False
assert data == [1, 2, 3]
assert alias == [1, 2, 3]
assert id(data) == old_id

print("rolled back:", data)

rolled back: [1, 2, 3]


### Solution notes

This pattern separates validation from commitment. The final assignment `seq[:] = working` preserves the identity of the original list while replacing its contents.

## Final Review Checklist

You should now be able to:

- Explain mutation vs rebinding.
- Predict alias behavior.
- Use `clear`, `append`, `extend`, `pop`, `insert`, and `reverse` correctly.
- Choose between in-place operations and copy-producing operations.
- Use slice assignment for controlled in-place replacement.
- Avoid shallow-copy surprises with nested lists.
- Mutate lists safely while preserving object identity.